# 02 — Accuracy by sentence type, prompt, model, and schema

This notebook addresses the tutor's core methodological questions: are models better at literal spatial language, metaphorical spatial language, or neither? Are some schema families easier than others? Does structured prompting help?

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
PARSED_PATH = DATA_DIR / "outputs" / "parsed_responses.jsonl"
RAW_PATH = DATA_DIR / "outputs" / "raw_responses.jsonl"
GOLD_PATH = DATA_DIR / "gold" / "sentences_v1.jsonl"

def read_jsonl(path: Path) -> pd.DataFrame:
    records = []
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                records.append(json.loads(line))
    return pd.DataFrame(records)

def safe_accuracy(df: pd.DataFrame, pred_col: str, gold_col: str):
    if pred_col not in df.columns or gold_col not in df.columns:
        return None
    sub = df[df[pred_col].notna() & df[gold_col].notna()]
    if sub.empty:
        return None
    return float((sub[pred_col].astype(str) == sub[gold_col].astype(str)).mean())

def pct(x):
    return "NA" if x is None or pd.isna(x) else f"{100*x:.1f}%"

In [ ]:
parsed = read_jsonl(PARSED_PATH)
structured = parsed[parsed["parse_status"].eq("parsed")].copy()
print(f"All parsed records: {len(parsed)}")
print(f"Structured parsed records for accuracy metrics: {len(structured)}")
display(structured.head())

In [ ]:
schema_acc = safe_accuracy(structured, "main_image_schema", "expected_schema_primary")
lm_acc = safe_accuracy(structured, "literal_or_metaphorical", "expected_literal_or_metaphorical")
print(f"Primary schema accuracy: {pct(schema_acc)}")
print(f"Literal/metaphorical accuracy: {pct(lm_acc)}")

In [ ]:
if not structured.empty:
    by_sentence_type = (
        structured.groupby("sentence_type")
        .apply(lambda g: pd.Series({
            "n": len(g),
            "schema_accuracy": safe_accuracy(g, "main_image_schema", "expected_schema_primary"),
            "literal_metaphorical_accuracy": safe_accuracy(g, "literal_or_metaphorical", "expected_literal_or_metaphorical")
        }))
        .reset_index()
    )
    display(by_sentence_type)

In [ ]:
if not structured.empty:
    by_prompt_sentence = (
        structured.groupby(["prompt_family", "sentence_type"])
        .apply(lambda g: pd.Series({
            "n": len(g),
            "schema_accuracy": safe_accuracy(g, "main_image_schema", "expected_schema_primary"),
            "literal_metaphorical_accuracy": safe_accuracy(g, "literal_or_metaphorical", "expected_literal_or_metaphorical")
        }))
        .reset_index()
    )
    display(by_prompt_sentence)

In [ ]:
if not structured.empty:
    plot_df = by_sentence_type.dropna(subset=["schema_accuracy"])
    ax = plot_df.plot(kind="bar", x="sentence_type", y="schema_accuracy", legend=False)
    ax.set_title("Primary schema accuracy by sentence type")
    ax.set_xlabel("Sentence type")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
if not structured.empty:
    by_schema = (
        structured.groupby("expected_schema_primary")
        .apply(lambda g: pd.Series({
            "n": len(g),
            "schema_accuracy": safe_accuracy(g, "main_image_schema", "expected_schema_primary"),
            "literal_metaphorical_accuracy": safe_accuracy(g, "literal_or_metaphorical", "expected_literal_or_metaphorical")
        }))
        .reset_index()
        .sort_values(["schema_accuracy", "n"], ascending=[False, False])
    )
    display(by_schema)

In [ ]:
if not structured.empty:
    display(pd.crosstab(structured["expected_schema_primary"], structured["main_image_schema"],
                        rownames=["gold"], colnames=["predicted"], dropna=False))

In [ ]:
if not structured.empty and "model_id" in structured.columns:
    by_model = (
        structured.groupby(["provider", "model_id"])
        .apply(lambda g: pd.Series({
            "n": len(g),
            "schema_accuracy": safe_accuracy(g, "main_image_schema", "expected_schema_primary"),
            "literal_metaphorical_accuracy": safe_accuracy(g, "literal_or_metaphorical", "expected_literal_or_metaphorical")
        }))
        .reset_index()
    )
    display(by_model)

## Interpretation logic

- **Literal ≈ metaphorical:** the prompt may make both uses similarly tractable, but this does not prove embodied cognition.
- **Literal > metaphorical:** the model may detect surface spatial structure more reliably than source–target mappings.
- **Metaphorical > literal:** the model may be responding to conventional metaphor-discourse cues or over-metaphorising.
- **Schema-family differences:** some schemas may be more lexically visible or more common in training data.